In [0]:
%pip install faker

In [0]:
dbutils.library.restartPython()

In [0]:
from faker import Faker
import random
import uuid

faker = Faker()

In [0]:
def generate_customer():
    customer = {
        "customer_id": str(uuid.uuid4()),
        "first_name": faker.first_name(),
        "last_name": faker.last_name(),
        "email": faker.email(),
        "country": faker.country(),
        "signup_date": faker.date_between(start_date="-2y", end_date="today").isoformat()
    }
    return customer

In [0]:
customers = [generate_customer() for _ in range(200)]
customers_df = spark.createDataFrame(customers)
customers_df.show(5)
print(f"Total customers: {customers_df.count()}")



In [0]:
def generate_product():
    categories = {
        "Electronics": ["Headphones", "Smartphone", "Laptop", "Smartwatch", "Tablet"],
        "Clothing": ["T-Shirt", "Jeans", "Jacket", "Sneakers", "Hat"],
        "Home & Kitchen": ["Blender", "Toaster", "Cookware Set", "Vacuum Cleaner", "Lamp"],
        "Books": ["Novel", "Cookbook", "Biography", "Textbook", "Comic"],
        "Sports": ["Yoga Mat", "Dumbbell Set", "Running Shoes", "Bicycle", "Tennis Racket"]
    }
    category = random.choice(list(categories.keys()))
    product = {
        "product_id": str(uuid.uuid4()),
        "product_name": random.choice(categories[category]) + " " + random.choice(["Pro", "Max", "Lite", "Standard", "Plus"]),
        "category": category,
        "price": round(random.uniform(5.0, 500.0), 2)
    }

    return product

In [0]:
products = [generate_product() for _ in range(50)]
products_df = spark.createDataFrame(products)
products_df.show(5)
print(f"Total products: {products_df.count()}")


In [0]:
client_id = dbutils.secrets.get(scope="ecommerce-project", key="sp-client-id")
tenant_id = dbutils.secrets.get(scope="ecommerce-project", key="sp-tenant-id")
client_secret = dbutils.secrets.get(scope="ecommerce-project", key="sp-client-secret")

spark.conf.set("fs.azure.account.auth.type.ecommercelakehouse01.dfs.core.windows.net", "OAuth")
spark.conf.set("fs.azure.account.oauth.provider.type.ecommercelakehouse01.dfs.core.windows.net", "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider")
spark.conf.set("fs.azure.account.oauth2.client.id.ecommercelakehouse01.dfs.core.windows.net", client_id)
spark.conf.set("fs.azure.account.oauth2.client.secret.ecommercelakehouse01.dfs.core.windows.net", client_secret)
spark.conf.set("fs.azure.account.oauth2.client.endpoint.ecommercelakehouse01.dfs.core.windows.net", f"https://login.microsoftonline.com/{tenant_id}/oauth2/token")

In [0]:
customers_df.write.format("parquet").mode("overwrite").save("abfss://bronze@ecommercelakehouse01.dfs.core.windows.net/customers/")
products_df.write.format("parquet").mode("overwrite").save("abfss://bronze@ecommercelakehouse01.dfs.core.windows.net/products/")

In [0]:
from datetime import datetime, timedelta

def generate_order(customers, products):
    customer = random.choice(customers)
    product = random.choice(products)
    quantity = random.randint(1, 5)
    
    order = {
        "order_id": str(uuid.uuid4()),
        "customer_id": customer["customer_id"],
        "product_id": product["product_id"],
        "order_timestamp": (datetime.now() - timedelta(days=random.randint(0, 90))).isoformat(),
        "quantity": quantity,
        "unit_price": product["price"],
        "total_amount": round(quantity * product["price"], 2),
        "payment_method": random.choice(["Credit Card", "Debit Card", "PayPal", "Cash on Delivery"])
    }
    return order

In [0]:
orders = [generate_order(customers, products) for _ in range(5000)]
orders_df = spark.createDataFrame(orders)
orders_df.show(5)
print(f"Total orders: {orders_df.count()}")

In [0]:
dbutils.fs.rm("abfss://bronze@ecommercelakehouse01.dfs.core.windows.net/orders/", recurse=True)

In [0]:
orders_df.write.format("parquet").mode("append").save("abfss://bronze@ecommercelakehouse01.dfs.core.windows.net/orders/")

In [0]:
verify_df = spark.read.format("parquet").load("abfss://bronze@ecommercelakehouse01.dfs.core.windows.net/orders/")
print(f"Total rows in bronze orders: {verify_df.count()}")